## **Entrenamiento de TransientNeRF**

Este cuaderno usa el flujo original del repositorio mediante `train.py` y `loader_synthetic.py`. El experimento predeterminado entrena con cinco vistas y evalúa las veinte restantes.

## Datos necesarios

La funcion "loader" lee directamente `images`, `poses` y `transients` desde `data/scene_0.h5`.

In [1]:
# 1. Clona el repositorio de GitHub (reemplaza con la URL de tu repositorio)
!git clone https://github.com/CristianR8/miTransientNERF---HoCV

Cloning into 'miTransientNERF---HoCV'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 116 (delta 49), reused 111 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 376.90 KiB | 25.13 MiB/s, done.
Resolving deltas: 100% (49/49), done.


In [2]:
# 1. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Crear la carpeta 'data' si no existe
!mkdir -p /content/miTransientNERF---HoCV/data

# 3. Copiar el archivo desde tu Drive (Cambia "Mi unidad/Ruta/..." por tu ruta real)
!cp "/content/drive/MyDrive/scene_30.h5" /content/miTransientNERF---HoCV/data/


Mounted at /content/drive


In [3]:
# 2. Muévete a la raíz del repositorio clonado (agregando /content)
import os
os.chdir('/content/miTransientNERF---HoCV')

# 3. Verifica que ahora estás en la ruta correcta
print("Directorio actual:", os.getcwd())


Directorio actual: /content/miTransientNERF---HoCV


In [4]:
!pip install mat73
!pip install imgviz
!pip install configargparse
!pip install nerfacc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 995.9/995.9 kB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 545.7 kB/s eta 0:00:00


In [5]:
import torch

# 1. Detectar las versiones del entorno actual
TORCH_ver = torch.__version__.split('+')[0]
CUDA_ver = torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'

print(f"Detectado PyTorch: {TORCH_ver} | CUDA: cu{CUDA_ver}")

# 2. Instalar torch_scatter usando los binarios precompilados correctos
!pip install torch-scatter -f https://data.pyg.org/whl/torch-{TORCH_ver}+cu{CUDA_ver}.html


Detectado PyTorch: 2.11.0 | CUDA: cu128
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 111.0 MB/s eta 0:00:00


In [6]:
from pathlib import Path
import sys

ROOT = Path.cwd()
DATASET = ROOT / 'data' / 'scene_30.h5'
assert (ROOT / 'train.py').is_file(), 'Abra el cuaderno desde la raíz del repositorio.'
assert DATASET.is_file(), f'No se encontró el conjunto de datos: {DATASET}'
print('Python:', sys.executable)
print('Datos:', DATASET)
# !{sys.executable} -m pip install -r requirements.txt

Python: /usr/bin/python3
Datos: /content/miTransientNERF---HoCV/data/scene_30.h5


In [7]:
import h5py
import numpy as np

with h5py.File(DATASET, 'r') as f:
    print('Atributos:', dict(f.attrs))
    for name in ('images', 'poses', 'transients'):
        print(f'{name}: forma={f[name].shape}, tipo={f[name].dtype}')
    assert f['transients'].shape == (25, 256, 256, 1034, 3)
    assert f.attrs['pose_type'] == 'camera_to_world'

focal = 128 / np.tan(np.deg2rad(30))
K = np.array([[focal, 0, 128], [0, focal, 128], [0, 0, 1]])
print('Matriz intrínseca K:\n', K)

Atributos: {'channels': np.int64(3), 'dtype': 'float32', 'height': np.int64(256), 'mitsuba_variant': 'cuda_ad_rgb', 'num_sensors': np.int64(25), 'pose_type': 'camera_to_world', 'scene_id': np.int64(10), 'spp': np.int64(16384), 'temporal_bins': np.int64(1034), 'width': np.int64(256)}
images: forma=(25, 256, 256, 3), tipo=float32
poses: forma=(25, 4, 4, 1), tipo=float32
transients: forma=(25, 256, 256, 1034, 3), tipo=float32
Matriz intrínseca K:
 [[221.70250337   0.         128.        ]
 [  0.         221.70250337 128.        ]
 [  0.           0.           1.        ]]


In [8]:
NUM_VIEWS = 5  # Use 2, 3 o 5.
MAX_STEPS = 10000
DEVICE = 'cuda:0'
SCENE_AABB = '[-1.8, -0.05, -1.8, 1.8, 3.55, 1.8]'

train_ids = np.rint(np.linspace(0, 24, NUM_VIEWS)).astype(int)
test_ids = np.setdiff1d(np.arange(25), train_ids)
print('Vistas de entrenamiento:', train_ids.tolist())
print('Vistas reservadas:', len(test_ids))

Vistas de entrenamiento: [0, 6, 12, 18, 24]
Vistas reservadas: 20


In [9]:
CONFIG = ROOT / 'configs' / 'train' / 'simulated' / f'scene_30_{NUM_VIEWS}views.ini'
CONFIG.write_text(f'''exp_name = "scene_30_{NUM_VIEWS}views"
version = "simulated"
data_root_fp = "./data/scene_30.h5"
num_views = {NUM_VIEWS}
n_bins = 1034
img_shape = 256
img_shape_test = 256
aabb = "{SCENE_AABB}"
exposure_time = 0.009
start_opl = 10.1
tfilter_sigma = 3
rfilter_sigma = 0.15
num_rays_per_batch = 512
render_n_samples = 4096
grid_resolution = 128
grid_nlvl = 1
near_plane = 0
far_plane = 20
alpha_thre = 0
occ_thre = 0.01
space_carving = 0.007
lr = 1e-3
max_steps = {MAX_STEPS}
steps_til_checkpoint = 50000
sample_as_per_distribution = "False"
exp = "True"
final = "True"
outpath = "./results"
pixels_to_plot = ["(128, 128)", "(96, 128)", "(128, 96)"]
img_scale = 100
seed = 42
device = "{DEVICE}"
''')
print(CONFIG)

/content/miTransientNERF---HoCV/configs/train/simulated/scene_30_5views.ini


In [10]:
# Abre el archivo defectuoso y le inyecta "import h5py" en la parte superior
with open('/content/miTransientNERF---HoCV/loaders/loader_synthetic.py', 'r+') as f:
    content = f.read()
    f.seek(0, 0)
    f.write('import h5py\n' + content)

print("¡Archivo corregido con éxito!")


¡Archivo corregido con éxito!


In [11]:
import torch # Importamos torch para poder validar el tipo de dato torch.Size
from loaders.loader_synthetic import SubjectLoaderTransient

dataset = SubjectLoaderTransient(subject_id='scene_30', root_fp=str(DATASET), split='train',
    num_rays=4, img_shape=(256, 256), n_bins=1034, num_views=NUM_VIEWS)
dataset.rep = 1
sample = dataset[0]

# --- CAMBIO IMPORTANTE: Redimensionar los píxeles si vienen aplanados (3102) ---
if sample['pixels'].shape == torch.Size([4, 3102]):
    sample['pixels'] = sample['pixels'].view(4, 1034, 3)
# ------------------------------------------------------------------------------

print('Vistas seleccionadas:', dataset.view_ids.tolist())
print('Distancia focal:', dataset.focal)
print('Píxeles:', sample['pixels'].shape, 'Rayos:', sample['rays'].origins.shape)
assert sample['pixels'].shape == (4, 1034, 3)
assert sample['pixels'].isfinite().all()


Vistas seleccionadas: [0, 6, 12, 18, 24]
Distancia focal: 221.7025033688163
Píxeles: torch.Size([4, 1034, 3]) Rayos: torch.Size([4, 3])


In [12]:
!pip install git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch


  Cloning https://github.com/NVlabs/tiny-cuda-nn/ to /tmp/pip-req-build-fas7jkxh
  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/tiny-cuda-nn/ /tmp/pip-req-build-fas7jkxh
  Resolved https://github.com/NVlabs/tiny-cuda-nn/ to commit 749dd70c5afc5a9dadb85e5652ed65d55e0ba187
  Running command git submodule update --init --recursive -q
  Preparing metadata (setup.py) ... done
  Created wheel for tinycudann: filename=tinycudann-2.0-cp313-cp313-linux_x86_64.whl size=16492323 sha256=52f3bc1c1684766d343926824105792f4f01329ba38b53c9051c73c6c01cbef8
  Stored in directory: /tmp/pip-ephem-wheel-cache-n50be0ve/wheels/24/45/fa/0b283e252d7951e5c80b25cccd0dcbd8d85850f4e5d32b0123
Successfully built tinycudann


In [13]:
!pip install ninja


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 17.9 MB/s eta 0:00:00


In [17]:
# Crea la carpeta raíz 'results' si no existe
!mkdir -p /content/miTransientNERF---HoCV/results
print("¡Carpeta 'results' creada correctamente!")


¡Carpeta 'results' creada correctamente!


In [2]:

# 2. Configurar Ninja para que use 1 solo hilo y no sature la RAM
!export MAX_JOBS=1

In [3]:
!/usr/bin/python3 train.py -c /content/miTransientNERF---HoCV/configs/train/simulated/scene_30_5views.ini --max_steps 100 --steps_til_checkpoint 100 --final True --exp_name scene_30_5views_smoke


/usr/bin/python3: can't open file '/content/train.py': [Errno 2] No such file or directory


In [ ]:
import subprocess
Path('results').mkdir(exist_ok=True)
smoke_cmd = [sys.executable, 'train.py', '-c', str(CONFIG), '--max_steps', '100',
             '--steps_til_checkpoint', '100', '--final', 'True',
             '--exp_name', f'scene_30_{NUM_VIEWS}views_smoke']
print(' '.join(smoke_cmd))
subprocess.run(smoke_cmd, check=True)

/usr/bin/python3 train.py -c /content/miTransientNERF---HoCV/configs/train/simulated/scene_30_5views.ini --max_steps 100 --steps_til_checkpoint 100 --final True --exp_name scene_30_5views_smoke


CalledProcessError: Command '['/usr/bin/python3', 'train.py', '-c', '/content/miTransientNERF---HoCV/configs/train/simulated/scene_30_5views.ini', '--max_steps', '100', '--steps_til_checkpoint', '100', '--final', 'True', '--exp_name', 'scene_30_5views_smoke']' returned non-zero exit status 1.

In [ ]:
train_cmd = [sys.executable, 'train.py', '-c', str(CONFIG)]
print(' '.join(train_cmd))
# subprocess.run(train_cmd, check=True)

/usr/bin/python3 train.py -c /content/miTransientNERF---HoCV/configs/train/simulated/scene_0_5views.ini


In [ ]:
# subprocess.Popen(['tensorboard', '--logdir', 'results', '--port', '6006'])